In [77]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin, clone, is_regressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils.estimator_checks import check_estimator
from sklearn.utils.validation import check_is_fitted, validate_data
from sklearn.model_selection import cross_validate

In [95]:
class ElectricRegressor(RegressorMixin, BaseEstimator):
    def __init__(self, estimator=None, pivot_col_index=-1):
        self.estimator = estimator
        self.pivot_col_index = pivot_col_index

    def __sklearn_tags__(self):
        tags = super().__sklearn_tags__()
        tags.target_tags.multi_output = True
        return tags

    def fit(self, X, y):
        assert self.estimator is not None, 'Must specify an estimator'
        assert is_regressor(self.estimator), 'Estimator must be a regressor'
        assert y.ndim >= 2 and y.shape[1] >= 2, 'y must have at least 2 columns'
        X, y = validate_data(self, X, y, multi_output=True)
        self.n_outputs_ = y.shape[1]

        idx = self.pivot_col_index if self.pivot_col_index >= 0 else y.shape[1] + self.pivot_col_index
        self.pivot_real_col_index_ = idx
        y1 = y[:, idx].reshape(-1, 1)
        mask = np.ones(y.shape[1], dtype=bool)
        mask[idx] = False
        y2 = y[:, mask]

        self.model1_ = clone(self.estimator)
        self.model1_.fit(X, y1.ravel())
        self.model2_ = clone(self.estimator)
        self.model2_.fit(np.hstack([X, y1]), y2)

        return self

    def predict(self, X):
        check_is_fitted(self, ['model1_', 'model2_'])

        in_data1 = validate_data(self, X, reset=False)
        predictions1 = self.model1_.predict(in_data1).reshape(-1, 1)
        in_data2 = np.hstack([in_data1, predictions1])
        predictions2 = self.model2_.predict(in_data2)

        n_samples = X.shape[0]
        predictions = np.zeros((n_samples, self.n_outputs_))
        predictions[:, self.pivot_real_col_index_] = predictions1.ravel()
        mask = np.ones(self.n_outputs_, dtype=bool)
        mask[self.pivot_real_col_index_] = False
        predictions[:, mask] = predictions2
        return predictions

In [96]:
model = ElectricRegressor(RandomForestRegressor(n_estimators=50, max_depth=15, min_samples_split=5, random_state=42))
check_estimator(model)
print(is_regressor(model))

AssertionError: y must have at least 2 columns

In [80]:
attributes = pd.read_csv('dataset/dataset_in.txt', index_col=0).drop(columns=['P2', 'Q2'])
objectives = pd.read_csv('dataset/dataset_out.txt', index_col=0).drop(columns=['I0_1', 'I0_12', 'I1_2', 'I2_3', 'I3_4', 'I4_5', 'I5_6', 'I7_8', 'I8_3', 'I8_9', 'I9_10', 'I10_11', 'I12_13'])
normalizer = MinMaxScaler(feature_range=(0, 1))
attributes = pd.DataFrame(normalizer.fit_transform(attributes), columns=attributes.columns)

In [89]:
model = ElectricRegressor(RandomForestRegressor(n_estimators=50, max_depth=15, min_samples_split=5, random_state=42))
results = cross_validate(model, attributes, objectives, scoring=['neg_mean_absolute_error', 'r2'], cv=10, n_jobs=-1, return_estimator=True, return_train_score=True)
print('MSE: ' + str(-results['test_neg_mean_absolute_error'].mean()))
print('R2: ' + str(results['test_r2'].mean()))

MSE: 1.9963075788567957
R2: 0.7946583002793547


In [84]:
example_in = attributes.iloc[30].to_frame().T
example_out = objectives.iloc[30].to_frame().T
example_out

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,PERD,QGEN7
day1_min0153,1.0,0.9918,0.9912,0.9909,0.9908,0.9908,0.9905,0.991,0.9909,0.9909,0.9909,0.9909,0.9933,0.9933,5.08,0.11


In [86]:
model = ElectricRegressor(RandomForestRegressor(n_estimators=50, max_depth=15, min_samples_split=5, random_state=42))
model.fit(attributes, objectives)
predictions = model.predict(example_in)
predictions